# 12. CAPM与Beta

**学习目标：**
1. 理解CAPM模型的推导逻辑和核心假设
2. 掌握Beta的经济含义与计算方法
3. 学会用滚动窗口估计时变Beta
4. 理解Alpha的来源与判断标准

**环境依赖：**
```bash
pip install numpy pandas matplotlib akshare scipy statsmodels
```

## 1. CAPM理论基础

### 1.1 什么是CAPM？

**CAPM (Capital Asset Pricing Model)** 是现代金融学的基石之一，由Sharpe(1964)、Lintner(1965)和Mossin(1966)独立提出。

核心思想：**资产的预期收益只与其系统性风险（Beta）相关**

### 1.2 CAPM的数学表达

$$E(R_i) = R_f + \beta_i \cdot [E(R_m) - R_f]$$

其中：
- $E(R_i)$：资产i的预期收益率
- $R_f$：无风险利率
- $\beta_i$：资产i的Beta系数
- $E(R_m) - R_f$：市场风险溢价 (Market Risk Premium)

### 1.3 CAPM的核心假设

| 假设 | 含义 | 现实偏离 |
|------|------|----------|
| 投资者理性 | 追求均值-方差最优 | 行为金融学挑战 |
| 同质预期 | 所有投资者预期相同 | 信息不对称 |
| 无摩擦市场 | 无交易成本、税收 | 现实有佣金、印花税 |
| 可无限制卖空 | 做空不受限制 | A股做空受限 |
| 单期投资 | 所有人同一投资期限 | 实际期限各异 |

## 2. Beta的经济含义

### 2.1 Beta的定义

$$\beta_i = \frac{Cov(R_i, R_m)}{Var(R_m)} = \rho_{i,m} \cdot \frac{\sigma_i}{\sigma_m}$$

Beta衡量的是：**资产对市场波动的敏感度**

### 2.2 Beta的解读

| Beta值 | 含义 | 典型行业 |
|--------|------|----------|
| β > 1 | 进攻型，波动大于市场 | 科技、券商 |
| β = 1 | 与市场同步 | 大盘蓝筹 |
| 0 < β < 1 | 防御型，波动小于市场 | 公用事业、消费 |
| β < 0 | 与市场反向 | 极少见，黄金有时 |
| β ≈ 0 | 与市场无关 | 现金类资产 |

### 2.3 Alpha的来源

根据CAPM，**Alpha是超额收益**：

$$\alpha_i = R_i - [R_f + \beta_i \cdot (R_m - R_f)]$$

**Alpha可能的来源：**
1. **市场无效** - 信息不对称、投资者非理性
2. **因子暴露** - 承担了CAPM未捕捉的风险（规模、价值、动量等）
3. **技能/运气** - 选股能力或随机噪声

⚠️ **重要提醒**：长期持续的正Alpha极其罕见，大多数「Alpha」其实是隐藏的Beta

## 3. 代码实现

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import akshare as ak
from scipy import stats
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# 中文显示配置
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'STHeiti']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 库加载成功')

### 3.1 数据准备

In [ ]:
# 获取50ETF及其成分股数据
print('正在获取数据...')

try:
    # 获取上证50成分股列表
    etf_spot = ak.fund_etf_spot_em()
    # 获取上证50指数（作为市场基准）
    market_df = ak.stock_zh_index_daily(symbol='sh000016')
    market_df = market_df[['date', 'close']].copy()
    market_df.columns = ['date', 'market_close']
    market_df['date'] = pd.to_datetime(market_df['date'])
    market_df = market_df.sort_values('date').reset_index(drop=True)
    
    # 获取几只典型50成分股
    stocks = {
        '600519': '贵州茅台',
        '601318': '中国平安',
        '600036': '招商银行',
        '601166': '兴业银行',
        '600276': '恒瑞医药',
        '601888': '中国中免',
        '600900': '长江电力',
        '601398': '工商银行',
        '600030': '中信证券',
        '603259': '药明康德'
    }
    
    stock_data = {}
    for code, name in stocks.items():
        try:
            df = ak.stock_zh_a_hist(symbol=code, period='daily', 
                                     start_date='20230101', end_date='20260601', 
                                     adjust='qfq')
            df = df[['日期', '收盘']].copy()
            df.columns = ['date', 'close']
            df['date'] = pd.to_datetime(df['date'])
            stock_data[code] = {'name': name, 'data': df}
            print(f'  ✓ {name}({code})')
        except Exception as e:
            print(f'  ✗ {name}({code}): {e}')
    
    print(f'\n成功获取 {len(stock_data)} 只股票数据')
    
except Exception as e:
    print(f'数据获取失败: {e}')
    print('使用模拟数据演示...')
    
    # 生成模拟数据
    np.random.seed(42)
    dates = pd.date_range('2023-01-01', '2026-06-01', freq='B')
    
    # 模拟市场收益
    market_returns = np.random.normal(0.0003, 0.012, len(dates))
    market_prices = 2500 * np.exp(np.cumsum(market_returns))
    
    market_df = pd.DataFrame({'date': dates, 'market_close': market_prices})
    
    # 模拟不同Beta的股票
    stock_configs = {
        '600519': ('贵州茅台', 0.8, 0.0005),   # 低Beta，正Alpha
        '601318': ('中国平安', 1.2, 0.0001),   # 高Beta
        '600036': ('招商银行', 0.9, 0.0002),   # 中低Beta
        '601166': ('兴业银行', 1.1, -0.0001),  # 中高Beta
        '600276': ('恒瑞医药', 0.7, 0.0003),   # 低Beta
        '601888': ('中国中免', 1.4, -0.0002),  # 高Beta
        '600900': ('长江电力', 0.5, 0.0002),   # 防御型
        '601398': ('工商银行', 0.6, 0.0001),   # 低Beta
        '600030': ('中信证券', 1.5, 0.0003),   # 券商高Beta
        '603259': ('药明康德', 1.3, -0.0001)   # 成长股高Beta
    }
    
    stock_data = {}
    for code, (name, beta, alpha) in stock_configs.items():
        returns = alpha + beta * market_returns + np.random.normal(0, 0.008, len(dates))
        prices = 100 * np.exp(np.cumsum(returns))
        stock_data[code] = {
            'name': name,
            'data': pd.DataFrame({'date': dates, 'close': prices})
        }
    
    print(f'✅ 模拟数据生成完成: {len(stock_data)} 只股票')

### 3.2 计算收益率与Beta

In [ ]:
# 合并所有数据，计算日收益率
def calculate_returns(market_df, stock_data):
    """计算市场和个股收益率"""
    # 市场收益率
    market_returns = market_df.set_index('date')['market_close'].pct_change().dropna()
    market_returns.name = 'market'
    
    # 个股收益率
    all_returns = pd.DataFrame({'market': market_returns})
    
    for code, info in stock_data.items():
        stock_returns = info['data'].set_index('date')['close'].pct_change().dropna()
        stock_returns.name = code
        all_returns = all_returns.join(stock_returns, how='inner')
    
    return all_returns.dropna()

returns_df = calculate_returns(market_df, stock_data)
print(f'收益率数据: {returns_df.shape[0]} 个交易日, {returns_df.shape[1]-1} 只股票')
print(f'时间范围: {returns_df.index[0].strftime("%Y-%m-%d")} ~ {returns_df.index[-1].strftime("%Y-%m-%d")}')
returns_df.head()

In [ ]:
# 方法1: 全样本Beta（简单线性回归）
def calculate_beta_full_sample(returns_df, stock_code):
    """全样本Beta计算"""
    X = returns_df['market']
    y = returns_df[stock_code]
    
    # 添加常数项（截距 = Alpha）
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()
    
    return {
        'alpha': model.params['const'] * 252,  # 年化Alpha
        'beta': model.params['market'],
        'r_squared': model.rsquared,
        'alpha_tvalue': model.tvalues['const'],
        'beta_tvalue': model.tvalues['market'],
        'alpha_pvalue': model.pvalues['const']
    }

# 计算所有股票的Beta
beta_results = {}
print('全样本Beta估计结果：')
print('=' * 70)
print(f'{"股票":<12} {"Beta":>8} {"年化Alpha":>12} {"R²":>8} {"Beta t值":>10} {"Alpha显著":>10}')
print('-' * 70)

for code in stock_data.keys():
    result = calculate_beta_full_sample(returns_df, code)
    beta_results[code] = result
    name = stock_data[code]['name']
    alpha_sig = '***' if result['alpha_pvalue'] < 0.01 else '**' if result['alpha_pvalue'] < 0.05 else '*' if result['alpha_pvalue'] < 0.1 else ''
    print(f'{name:<10} {result["beta"]:>8.3f} {result["alpha"]:>11.2%} {result["r_squared"]:>8.3f} {result["beta_tvalue"]:>10.2f} {alpha_sig:>10}')

print('=' * 70)
print('注: Alpha为年化超额收益, *** p<0.01, ** p<0.05, * p<0.1')

### 3.3 滚动窗口Beta（Beta的时变性）

In [ ]:
# 方法2: 滚动窗口Beta
def calculate_rolling_beta(returns_df, stock_code, window=60):
    """滚动窗口Beta计算"""
    rolling_beta = []
    rolling_alpha = []
    dates = []
    
    for i in range(window, len(returns_df)):
        window_data = returns_df.iloc[i-window:i]
        X = sm.add_constant(window_data['market'])
        y = window_data[stock_code]
        model = sm.OLS(y, X).fit()
        
        rolling_beta.append(model.params['market'])
        rolling_alpha.append(model.params['const'] * 252)  # 年化
        dates.append(returns_df.index[i])
    
    return pd.DataFrame({
        'date': dates,
        'beta': rolling_beta,
        'alpha': rolling_alpha
    }).set_index('date')

# 计算滚动Beta
window = 60  # 60个交易日 ≈ 3个月
rolling_betas = {}

for code in stock_data.keys():
    rolling_betas[code] = calculate_rolling_beta(returns_df, code, window)

# 可视化
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 选择几只典型股票展示
display_stocks = ['600519', '601318', '600900', '600030']

# 滚动Beta
ax1 = axes[0, 0]
for code in display_stocks:
    ax1.plot(rolling_betas[code].index, rolling_betas[code]['beta'], 
             label=f'{stock_data[code]["name"]}', linewidth=1.5)
ax1.axhline(y=1, color='gray', linestyle='--', alpha=0.5, label='β=1')
ax1.set_title(f'滚动Beta走势 (窗口={window}日)', fontsize=14)
ax1.set_ylabel('Beta')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)

# 滚动Alpha
ax2 = axes[0, 1]
for code in display_stocks:
    ax2.plot(rolling_betas[code].index, rolling_betas[code]['alpha'] * 100, 
             label=f'{stock_data[code]["name"]}', linewidth=1.5)
ax2.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax2.set_title(f'滚动年化Alpha (%)', fontsize=14)
ax2.set_ylabel('Alpha (%)')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)

# Beta分布
ax3 = axes[1, 0]
beta_values = [rolling_betas[code]['beta'].mean() for code in stock_data.keys()]
names = [stock_data[code]['name'] for code in stock_data.keys()]
colors = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(beta_values)))

sorted_idx = np.argsort(beta_values)
bars = ax3.barh([names[i] for i in sorted_idx], [beta_values[i] for i in sorted_idx], color=colors)
ax3.axvline(x=1, color='red', linestyle='--', alpha=0.7, label='β=1')
ax3.set_title('各股票平均Beta对比', fontsize=14)
ax3.set_xlabel('Beta')
ax3.legend()
ax3.grid(True, alpha=0.3, axis='x')

# Alpha vs Beta散点图
ax4 = axes[1, 1]
avg_betas = [rolling_betas[code]['beta'].mean() for code in stock_data.keys()]
avg_alphas = [rolling_betas[code]['alpha'].mean() * 100 for code in stock_data.keys()]

ax4.scatter(avg_betas, avg_alphas, s=100, c='steelblue', alpha=0.7)
for i, code in enumerate(stock_data.keys()):
    ax4.annotate(stock_data[code]['name'], (avg_betas[i], avg_alphas[i]),
                 xytext=(5, 5), textcoords='offset points', fontsize=9)
ax4.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax4.axvline(x=1, color='gray', linestyle='--', alpha=0.5)
ax4.set_title('Alpha vs Beta 散点图', fontsize=14)
ax4.set_xlabel('平均Beta')
ax4.set_ylabel('平均年化Alpha (%)')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('capm_beta_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ 图表已保存为 capm_beta_analysis.png')

### 3.4 按Beta分组对比

In [ ]:
# 按Beta分三组：高Beta、中Beta、低Beta
def categorize_by_beta(beta_results):
    """按Beta分组"""
    betas = {code: result['beta'] for code, result in beta_results.items()}
    sorted_stocks = sorted(betas.items(), key=lambda x: x[1])
    
    n = len(sorted_stocks)
    low = [s[0] for s in sorted_stocks[:n//3]]
    mid = [s[0] for s in sorted_stocks[n//3:2*n//3]]
    high = [s[0] for s in sorted_stocks[2*n//3:]]
    
    return {'低Beta': low, '中Beta': mid, '高Beta': high}

groups = categorize_by_beta(beta_results)

# 计算各组的平均表现
print('按Beta分组分析：')
print('=' * 70)

group_stats = {}
for group_name, codes in groups.items():
    group_betas = [beta_results[c]['beta'] for c in codes]
    group_alphas = [beta_results[c]['alpha'] for c in codes]
    group_names = [stock_data[c]['name'] for c in codes]
    
    group_stats[group_name] = {
        'stocks': group_names,
        'avg_beta': np.mean(group_betas),
        'avg_alpha': np.mean(group_alphas),
        'std_beta': np.std(group_betas)
    }
    
    print(f'\n【{group_name}】')
    print(f'  成分股: {", ".join(group_names)}')
    print(f'  平均Beta: {np.mean(group_betas):.3f} ± {np.std(group_betas):.3f}')
    print(f'  平均年化Alpha: {np.mean(group_alphas):.2%}')

print('\n' + '=' * 70)

# 可视化分组对比
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# 各组累计收益对比
ax1 = axes[0]
for group_name, codes in groups.items():
    group_cum_returns = []
    for code in codes:
        cum_returns = (1 + returns_df[code]).cumprod()
        group_cum_returns.append(cum_returns)
    avg_cum = pd.concat(group_cum_returns, axis=1).mean(axis=1)
    ax1.plot(avg_cum.index, avg_cum.values, label=group_name, linewidth=2)

# 市场基准
market_cum = (1 + returns_df['market']).cumprod()
ax1.plot(market_cum.index, market_cum.values, label='市场', linewidth=2, linestyle='--', color='black')

ax1.set_title('各Beta组累计收益对比', fontsize=14)
ax1.set_ylabel('累计净值')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Beta箱线图
ax2 = axes[1]
beta_data = [rolling_betas[code]['beta'].values for code in stock_data.keys()]
names = [stock_data[code]['name'] for code in stock_data.keys()]
bp = ax2.boxplot(beta_data, labels=names, vert=True)
ax2.axhline(y=1, color='red', linestyle='--', alpha=0.5)
ax2.set_title('各股票Beta分布', fontsize=14)
ax2.set_ylabel('Beta')
ax2.tick_params(axis='x', rotation=45)
ax2.grid(True, alpha=0.3)

# 风险-收益散点图
ax3 = axes[2]
for group_name, codes in groups.items():
    group_returns = [returns_df[code].mean() * 252 * 100 for code in codes]
    group_vol = [returns_df[code].std() * np.sqrt(252) * 100 for code in codes]
    ax3.scatter(group_vol, group_returns, s=100, label=group_name, alpha=0.7)

# 市场点
market_ret = returns_df['market'].mean() * 252 * 100
market_vol = returns_df['market'].std() * np.sqrt(252) * 100
ax3.scatter([market_vol], [market_ret], s=150, c='black', marker='*', label='市场')

ax3.set_title('风险-收益图 (年化)', fontsize=14)
ax3.set_xlabel('年化波动率 (%)')
ax3.set_ylabel('年化收益率 (%)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('beta_group_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ 分组对比图已保存为 beta_group_comparison.png')

### 3.5 CAPM回归可视化

In [ ]:
# 个股与市场的散点图 + 回归线
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

selected_stocks = ['600519', '601318', '600036', '600900', '600030', '601888']

for idx, code in enumerate(selected_stocks):
    ax = axes[idx // 3, idx % 3]
    
    # 散点图
    ax.scatter(returns_df['market'] * 100, returns_df[code] * 100, 
               alpha=0.3, s=20, c='steelblue')
    
    # 回归线
    beta = beta_results[code]['beta']
    alpha = beta_results[code]['alpha'] / 252  # 日度Alpha
    x_range = np.linspace(returns_df['market'].min() * 100, 
                          returns_df['market'].max() * 100, 100)
    ax.plot(x_range, alpha * 100 + beta * x_range, 'r-', linewidth=2, 
            label=f'β={beta:.2f}, α={beta_results[code]["alpha"]:.1%}/年')
    
    # 45度线
    ax.plot(x_range, x_range, 'k--', alpha=0.3, label='β=1')
    
    ax.set_title(f'{stock_data[code]["name"]}', fontsize=12)
    ax.set_xlabel('市场收益 (%)')
    ax.set_ylabel('个股收益 (%)')
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('CAPM回归: 个股收益 vs 市场收益', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('capm_regression_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('✅ CAPM回归散点图已保存')

## 4. 实战任务：50ETF成分股Beta分析

### 4.1 完整分析流程

In [ ]:
# 汇总表格
summary_data = []
for code in stock_data.keys():
    result = beta_results[code]
    rolling = rolling_betas[code]
    
    summary_data.append({
        '股票代码': code,
        '股票名称': stock_data[code]['name'],
        '全样本Beta': result['beta'],
        '滚动Beta均值': rolling['beta'].mean(),
        '滚动Beta标准差': rolling['beta'].std(),
        'Beta变异系数': rolling['beta'].std() / rolling['beta'].mean(),
        '年化Alpha': result['alpha'],
        'Alpha显著性': result['alpha_pvalue'],
        'R²': result['r_squared'],
        '年化波动率': returns_df[code].std() * np.sqrt(252),
        '年化收益': returns_df[code].mean() * 252
    })

summary_df = pd.DataFrame(summary_data)

# 按Beta排序
summary_df = summary_df.sort_values('全样本Beta', ascending=False)

# 格式化输出
print('50ETF成分股CAPM分析汇总表')
print('=' * 100)
print(f'{"股票":<12} {"Beta":>8} {"Beta波动":>10} {"年化Alpha":>12} {"Alpha显著":>12} {"R²":>8} {"年化收益":>12} {"年化波动":>12}')
print('-' * 100)

for _, row in summary_df.iterrows():
    alpha_sig = '***' if row['Alpha显著性'] < 0.01 else '**' if row['Alpha显著性'] < 0.05 else '*' if row['Alpha显著性'] < 0.1 else ''
    print(f'{row["股票名称"]:<10} {row["全样本Beta"]:>8.3f} {row["滚动Beta标准差"]:>10.3f} {row["年化Alpha"]:>11.2%} {alpha_sig:>12} {row["R²"]:>8.3f} {row["年化收益"]:>11.2%} {row["年化波动"]:>11.2%}')

print('=' * 100)

# 保存结果
summary_df.to_excel('capm_beta_results.xlsx', index=False)
print('\n✅ 详细结果已保存到 capm_beta_results.xlsx')

## 5. 关键洞察

In [ ]:
print('📊 关键洞察：')
print('=' * 70)

print('\n1️⃣ Beta的时变性：')
print('   - Beta不是固定不变的，会随市场环境变化')
print('   - 牛市中，高Beta股更Beta；熊市中，Beta可能收缩')
print('   - 使用滚动窗口比全样本更贴近实际')

print('\n2️⃣ Alpha的稀缺性：')
avg_alpha = summary_df['年化Alpha'].mean()
print(f'   - 平均年化Alpha: {avg_alpha:.2%}')
print('   - 大部分Alpha统计不显著 (p>0.1)')
print('   - 即使显著，也要考虑是否是数据挖掘偏差')

print('\n3️⃣ 高Beta不等于高收益：')
high_beta_return = summary_df[summary_df['全样本Beta'] > 1.2]['年化收益'].mean()
low_beta_return = summary_df[summary_df['全样本Beta'] < 0.8]['年化收益'].mean()
print(f'   - 高Beta组(>1.2)平均年化收益: {high_beta_return:.2%}')
print(f'   - 低Beta组(<0.8)平均年化收益: {low_beta_return:.2%}')
print('   - 「低Beta异象」在全球市场普遍存在')

print('\n4️⃣ CAPM的局限性：')
print('   - 单因子模型，忽略了规模、价值、动量等因子')
print('   - R²通常不高，说明个股特异性风险很大')
print('   - 现实中，多因子模型更有效 (下节课讲Fama-French)')

print('\n' + '=' * 70)

## 6. 小结与验收标准

### 本节要点

| 概念 | 核心内容 |
|------|----------|
| CAPM公式 | E(Ri) = Rf + βi × (E(Rm) - Rf) |
| Beta含义 | 资产对市场波动的敏感度，β>1进攻型，β<1防御型 |
| Alpha含义 | 超额收益，CAPM认为长期Alpha应为0 |
| 滚动Beta | 捕捉Beta的时变性，比全样本更实用 |
| 低Beta异象 | 低Beta股票往往有更高的风险调整收益 |

### 验收标准 Checklist

- [x] 能用滚动窗口估计Beta
- [x] 理解Beta的时变性
- [x] 能解释Alpha的来源

### 下节预告

**序号14: Fama-French三因子模型**
- 规模因子SMB、价值因子HML
- 因子收益率的计算方法
- 多因子回归分析